<a href="https://colab.research.google.com/github/Aryankumar507/AI-Image-Restoration/blob/main/comment_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

Now Lets make some text

In [ ]:
sentences = [
    # 50 Positive sentences
    "I love this product",
    "This movie made me smile",
    "Service was friendly and quick",
    "Today felt bright and happy",
    "This is the best day",
    "Absoulutely fantastic experience",
    "I had a wonderful time",
    "The food was delicious",
    "What a beautiful day it is",
    "Feeling great today",
    "This is truly amazing",
    "I am so happy with the result",
    "Everything turned out perfectly",
    "Such an enjoyable moment",
    "This made my day",
    "Absolutely love it",
    "Highly recommend this experience",
    "So much fun",
    "A fantastic achievement",
    "I'm very impressed",
    "The best decision ever",
    "Pure joy and happiness",
    "Simply wonderful",
    "Couldn't ask for more",
    "Delightful and pleasant",
    "A truly positive outcome",
    "Exceeded all expectations",
    "Brilliant work",
    "Feeling blessed",
    "A cheerful atmosphere",
    "Outstanding performance",
    "This brings me so much comfort",
    "A perfect evening",
    "Grateful for this opportunity",
    "Wonderful memories made",
    "Such a kind gesture",
    "I'm optimistic about the future",
    "This is exactly what I needed",
    "A delightful surprise",
    "So much positivity around", # Corrected missing comma
    "Feeling inspired and motivated",
    "This outcome is fantastic",
    "A truly rewarding endeavor",
    "Everything is perfect today",
    "Full of gratitude and joy",
    "What a pleasant surprise",
    "So happy with the results",
    "This has made my week",
    "An absolute delight",
    "Positively thrilled by this",
    # 50 Negative sentences
    "This product is terrible",
    "The movie was incredibly boring",
    "Service was slow and unhelpful",
    "Today felt gloomy and sad",
    "This is the worst day",
    "Absolutely awful experience",
    "I had a miserable time",
    "The food was disgusting",
    "What a terrible day it is",
    "Feeling awful today",
    "This is truly disappointing",
    "I am so unhappy with the result",
    "Everything went wrong",
    "Such a frustrating moment",
    "This ruined my day",
    "Absolutely hate it",
    "Would not recommend this experience",
    "So much wasted time",
    "A catastrophic failure",
    "I'm very displeased",
    "This is a complete disaster",
    "Extremely dissatisfied",
    "A waste of money and time",
    "Never again",
    "Beyond frustrated",
    "Terribly disappointed",
    "This is unacceptable",
    "A very poor choice",
    "I regret this purchase",
    "Feeling utterly miserable",
    "What a letdown",
    "This makes me angry",
    "A truly horrendous experience",
    "Could not be worse",
    "I despise this",
    "So unbelievably bad",
    "A complete failure",
    "I wish I hadn't come",
    "This is a nightmare",
    "Nothing went right",
    "Worst ever",
    "Absolutely no redeeming qualities",
    "This is a scam",
    "Feeling robbed",
    "A horrible mistake",
    "Deeply regret it",
    "Simply terrible",
    "Disappointing and sad",
    "Not worth the effort",
    "Lost all hope"
]
lables = [1]*50 + [0]*50
lables = np.array(lables)


In [ ]:
vocab_size = 6000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(sentences)
maxlen = max(len(s) for s in sequences)
x = pad_sequences(sequences, maxlen=maxlen,padding = 'post')
y = lables

In [ ]:
maxlen

7

In [ ]:
x[0]

array([ 6, 33,  2, 34,  0,  0,  0], dtype=int32)

In [ ]:
embed_dimen = 16
rnn_units = 8

In [ ]:
inp = Input(shape = (maxlen,),dtype = "int32",name = 'input')
embedded_inputs = Embedding(input_dim=vocab_size,output_dim=embed_dimen,mask_zero=True,name = 'embedding')(inp)
rnn = SimpleRNN(rnn_units,return_sequences=True,return_state=True,name = 'simple_rnn')
rnn_output_sequences, rnn_last_state = rnn(embedded_inputs)
out = Dense(1,activation = 'sigmoid',name = 'output')(rnn_last_state)
model = Model(inputs = inp,outputs = out)
model.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 7)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 7, 16)     │     96,000 │ input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 7)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 7, 8),    │        200 │ embedding[0][0],  │
│ (SimpleRNN)         │ (None, 8)]        │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │          9 │ simple_rnn[0][1]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 96,209 (375.82 KB)

 Trainable params: 96,209 (375.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Re-evaluate x and y to ensure they are NumPy arrays just before training
# (These lines are copied from cell HGwTpNEUwSgM to ensure correctness)
vocab_size = 6000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
maxlen = max(len(s) for s in sequences) # Recalculate maxlen based on current sentences
x = pad_sequences(sequences, maxlen=maxlen,padding = 'post')
y = lables.astype(float) # Ensure y is float for binary_crossentropy

model.fit(x, y, epochs=32, batch_size=8, verbose=1)

Epoch 1/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9700 - loss: 0.4594
Epoch 2/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9700 - loss: 0.3932
Epoch 3/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9700 - loss: 0.3321
Epoch 4/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9900 - loss: 0.2728
Epoch 5/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9900 - loss: 0.2248
Epoch 6/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.1858
Epoch 7/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1562
Epoch 8/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1330
Epoch 9/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.1154
Epoch 10/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1000
Epoch 11/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0893
Epoch 12/32
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - l